## This demo presents the implementation for the RSPY-323 story.

In [1]:
import requests
import os
import json
import pprint
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
user = os.environ["JUPYTERHUB_USER"] if cluster_mode else os.environ["RSPY_HOST_USER"]
auxip_client, cadip_client, catalog_client, staging_client = init_demo(owner_id = user)
if os.getenv("RSPY_LOCAL_MODE") == "1":
    href = "http://rs-server-adgs:8000"
    href_staging = "http://rs-server-staging:8000"
else:
    href = os.environ["RSPY_WEBSITE"]
    href_staging = "https://dev-rspy.esa-copernicus.eu"
    session.cookies.set ("session", os.environ["RSPY_OAUTH2_COOKIE"])

adgs_collection_id = "adgs"

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000


In [2]:
# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/89c7479624374cadbf26fd60109f465b/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [3]:
# Create a test collection 
collection = create_test_collection()

### Check the added collection with the rs-server-catalog. This collection should be empty.

In [4]:
# Check the catalog for agrosu_my_test_collection
catalog_collection = catalog_client.get_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert isinstance(catalog_collection, CollectionClient)
assert len(list(catalog_collection.get_items())) == 0
print(f"No items found in the '{TEST_COLLECTION}' collection")

No items found in the 'my_test_collection' collection


### Creating a staging body to start the staging process

In [5]:
items_collection = auxip_client.search(max_items = 10, collections = [adgs_collection_id])
assert len(items_collection) > 0

### Start the staging process for adgs

In [6]:
staging_resp_list = []
for items in items_collection:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))

timeout = 120
started_job_id_list = []

for resp in staging_resp_list:
    started_job_id_list.append(resp["jobID"])
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        # TODO: to replace with the following commented line after the rs-server-staging update
        ###job_info = staging_client.get_job_info(resp["jobID"])
        job_info = staging_client.get_job_info(resp["jobID"])
        pprint.PrettyPrinter(indent=4).pprint(job_info)
        print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

{   'created': datetime.datetime(2025, 3, 25, 21, 24, 14, tzinfo=<isodate.tzinfo.Utc object at 0x7f75edf911d0>),
    'jobID': 'ea52dd5f-4f26-43d6-869b-b00e6b5d3769',
    'message': 'Finished',
    'processID': 'staging',
    'progress': 100,
    'started': datetime.datetime(2025, 3, 25, 21, 24, 14, tzinfo=<isodate.tzinfo.Utc object at 0x7f75edf911d0>),
    'status': 'successful',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 21, 24, 14, tzinfo=<isodate.tzinfo.Utc object at 0x7f75edf911d0>)}


 ----- Job COMPLETED 

{   'created': datetime.datetime(2025, 3, 25, 21, 24, 14, tzinfo=<isodate.tzinfo.Utc object at 0x7f75edf911d0>),
    'jobID': '9726e75f-9d44-448a-adce-22ebe7fd72aa',
    'message': 'Finished',
    'processID': 'staging',
    'progress': 100,
    'started': datetime.datetime(2025, 3, 25, 21, 24, 14, tzinfo=<isodate.tzinfo.Utc object at 0x7f75edf911d0>),
    'status': 'successful',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 21, 

### Check the catalog for the present items.

In [7]:
# Check the catalog for agrosu_my_test_collection
catalog_collection = catalog_client.get_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert isinstance(catalog_collection, CollectionClient)
assert len(list(catalog_collection.get_items())) > 0
for item in catalog_collection.get_items():
    print(f"Item {item.id} has {len(item.assets)} assets")     

Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231007T062732_V20231007T062732_20231007T062732.EOF has 1 assets


### Delete one item from the collection

In [8]:
item_to_delete = "S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF"
result = catalog_client.remove_item(collection_id=TEST_COLLECTION, owner_id=user, item_id=item_to_delete)
assert result.json()["deleted item"] == "S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF"
pp.pprint(result.json())

{'deleted item': 'S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF'}


### Delete the whole collection

In [9]:
result = catalog_client.remove_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())

{'deleted collection': 'my_test_collection'}
